# End-to-end simulation of observation and pipeline reduction of HR 8799 d

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [github](https://github.com/oirlab/HISPEC_DRP).</span>


This notebook shows an example of simulating a single BSPEC exposure of an off guiding axis observation of HR 8799 d. 

The notebook simulates a BSPEC rate map using a combination of a Phoenix template (approximately similar to HR 8799 A) and Sonora bobcat template (approximately similar to HR 8799 d). The rate map is used to build a `RampModel`, then run `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline`.

Documentation relating to the data products produced by each pipeline stage can be found in the [API documentation](https://oirlab.github.io/HISPEC_DRP/api/generated/hispecdrp.datamodels.html). RSPEC data can be simulated by changing the SPECTROGRAPH keyword. In nominal instrument operation, both BSPEC and RSPEC exposures are obtained at the same time.

---

## Setup

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/path/to/hr8799d_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])

## 1. Sync / verify KOA caldb (BSPEC)

In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])

## 2. Creating input spectra from templates

In the next cells, we will normalize the planet spectra to the correct magnitude, and add the stellar spectrum as a speckle at the expected contrast for the Keck PSF. We will also create the spectrum for a non-science speckle fiber which only contains a reduced portion of the stellar flux.

First, load the template spectra and normalize to approximate J band magnitudes.

In [ ]:
hr8799A_magnitude_J_vega = 5.38
hr8799A_magnitude_J_AB = hr8799A_magnitude_J_vega + 0.91

hr8799d_magnitude_J_vega = 15.61 # Zurlo et al. 2016
hr8799d_magnitude_J_AB = hr8799d_magnitude_J_vega + 0.91

from hispecdrp.utils.spectral_template_utils import load_btsettl_spectrum
from hispecdrp.simulations.e2e_support import templates_data_dir

template_dir = templates_data_dir()+'/'
hr8799A_template_filename = 'lte07400-4.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5'
hr8799d_template_filename = 'sp_t1100g178nc_m+0.5.h5'

hr8799A_wvs, hr8799A_flam = load_btsettl_spectrum(
    filepath=template_dir+hr8799A_template_filename,
    mag_ab=hr8799A_magnitude_J_AB,
    filter_band='J'
)

hr8799d_wvs, hr8799d_flam = load_btsettl_spectrum(
    filepath=template_dir+hr8799d_template_filename,
    mag_ab=hr8799d_magnitude_J_AB,
    filter_band='J'
)

Now we will scale the stellar spectrum based on its separation from the star in the science and speckle fibers. A helper function is provided to estimate the speckle contrast as a function of separation. For simplicity, the speckle contrast is assumed to be constant with wavelength.

In [ ]:
import numpy as np

def estimated_speckle_contrast_at_separation(separation):
    # derived from linear fit to Ruffio et al. 2026 figure 17, Keck/OSIRIS in K band
    # separation must be in arcseconds

    m = -2.480700421412601
    b = -9.47882696118136
    log_sep = np.log(separation)
    log_contrast = m * log_sep + b

    return np.exp(log_contrast)

In [ ]:
# here, the speckle spectrum in the speckle fiber will be identical to speckle spectrum injected into the science fiber 
# this is adjustable by changing the fiber separations below
separation_hr8799d = 0.7 # arcseconds
separation_speckle_fiber = 0.7 # arcseconds

speckle_contrast_hr8799d = estimated_speckle_contrast_at_separation(separation_hr8799d)
speckle_contrast_speckle_fiber = estimated_speckle_contrast_at_separation(separation_speckle_fiber)

stellar_speckle_planet_flam = hr8799A_flam * speckle_contrast_hr8799d
stellar_speckle_fiber_flam =  hr8799A_flam * speckle_contrast_speckle_fiber

# fiber 1 spectrum is planet + stellar speckle, we will interpolate the planet spectrum to the stellar wavelength grid
fiber1_flam = np.interp(hr8799A_wvs, hr8799d_wvs, hr8799d_flam) + stellar_speckle_planet_flam
# fiber 2 spectrum is just stellar speckles
fiber2_flam = stellar_speckle_fiber_flam
# no fiber 3 spectrum needs to be constructed since it will just be sky


## 3. Simulated rate map from 1D spectra → `RampModel`

We will construct a dictionary of the spectra, RVs, and other inputs for each fiber, create an idealized rate map, which will then be converted to an initial ramp cube with artificial detector and poisson noise. Because HR 8779 d is near edge on, we will make the simplifying assumption that is has an identical radial velocity to the star. If not using this assumption, we would have to apply RVs separately to the template spectra before combining, and not use the included RV functionality below. 

1. Fiber 1 is the on the planet, which includes stellar speckles, and sky. 
2. Fiber 2 is a sample of the stellar speckles. Here we assume the speckles are the same as on fiber 1, but of course this will not generally be the case. 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind
- instrument thermal background

Because long (10+ min) exposures generate extremely memory intensive ramp cubes (10s of GB), we have artificially increased the read time ~10x the expected read time to reduce memory requirements.  <span style="color:red">WARNING: </span> simulating ramp cubes with large numbers of reads may crash your system if they exceed available memory.

In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1,2,3]
READ_TIME = 17 # seconds

# Injected stellar RV (km/s). Default path densifies the 1D template, Doppler-shifts
# it, paints with ZEMAX, and attaches the fixed packaged line-ZP WAVE (no WAVE×(1+z)).
STELLAR_RV = -10.41

obs_jd = 2461693.7083 # arbitrary, october 15 2027 05:00 UTC

exposure_time = 600 # seconds
n_reads = int(exposure_time/READ_TIME)


In [ ]:

# Create a UTR map from a controlled seed and template spectrum
# The user can normalize to a specific magnitude (AB mag) and 2MASS filter
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    fiber_spectra={
        1:(hr8799A_wvs, fiber1_flam),
        2:(hr8799A_wvs, fiber2_flam),
    },
    filter_band="J",  # BSPEC (yJ)
    n_reads=n_reads,
    readtime=READ_TIME,
    seed=0,
    radial_velocity_km_s=STELLAR_RV, # no dictionary needed since it is the same for both fibers
    barycentric_velocity_km_s=0.0,
    inject_rv_via="spectrum",
    sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
    apply_sky=[1,2,3],
    apply_e2e_throughput=[1,2,3],
    mjd_start=obs_jd-2400000.5,
    save_ramp=True,
    overwrite_ramp=True,
    poisson_noise=True,
    reuse_stage1_sims=True,
    register_stage1_sims=False
)

print(f"fibers: {FIBERS}")
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")
print(f"wavesol note: {getattr(getattr(cals.get('wavesol'), 'meta', None), 'note', None)}")

In [ ]:

# add fiber keywords
setattr(ramp.meta, 'object_fiber1', 'SCIENCE')
setattr(ramp.meta, 'object_fiber2', 'SPECKLE')
setattr(ramp.meta, 'object_fiber3', 'SKY')

# add exposure time keyword
setattr(ramp.meta, 'exposure_time', exposure_time)
setattr(ramp.meta, 'exposure_type', 'SPEC')

# set target keywords
setattr(ramp.meta, 'target_gaia_id', '2832463659640297472')
setattr(ramp.meta, 'target_name', 'HR 8799')
setattr(ramp.meta, 'target_ra', 346.8696250)
setattr(ramp.meta, 'target_dec', 21.1342528)

# save again
ramp.save(filepath=os.path.join(OUT, 'HB.20271015.17997.12_raw.fits'))

## 4. Stage 0 → 1 → 2 via pipeline classes

In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

os.environ["HISPEC_DRP_VIZIER_MIRRORS"] = "vizier.cfa.harvard.edu"

l0 = Stage0Pipeline(output_dir=OUT, dataset=ramp).apply()

do_fibers = [str(f) for f in FIBERS]
# Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
stage1_args = {
    "spec_extract": {"do_fibers": do_fibers},
    "set_wavesol": {"wavesol": cals["wavesol"]},
}
l1 = Stage1Pipeline(output_dir=OUT, primitives=stage1_args, dataset=l0).apply()

injected_m_s = float(STELLAR_RV) * 1000.0
l2 = Stage2Pipeline(
    output_dir=OUT,
    dataset=l1,
    primitives={
        "blaze_corr": {"do_fibers": do_fibers},
        "stellar_rv": {"run": False},
        "sky_sub": {"run": True},
    },
).apply()

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")


## 5. Set of high-contrast observations

We now simulate a set of multiple exposures spanning 1 hour of an HR 8799 d analog.

### 5.1 Simulating rate maps from an initial 1D spectrum and varying RV

1. Fiber 1 is the on the planet, which includes stellar speckles, and sky. 
2. Fiber 2 is a sample of the stellar speckles. Here we assume the speckles are the same as on fiber 1, but of course this will not generally be the case. 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind
- instrument thermal background

In [ ]:
import numpy as np

FIBERS = [1,2,3]

# Injected stellar RV (km/s). Default path densifies the 1D template, Doppler-shifts
# it, paints with ZEMAX, and attaches the fixed packaged line-ZP WAVE (no WAVE×(1+z)).
STELLAR_RV = -10.41

# observation parameters
starting_jd = 2461693.7083 # arbitrary, october 15 2027 05:00 UTC
READ_TIME = 17 # seconds
exposure_time = 600 # seconds
n_reads = int(exposure_time/READ_TIME)
total_exposures = 6

# creating array of observation start times
obs_jd_arr = starting_jd + np.arange(0,total_exposures*exposure_time/86400,exposure_time/86400)

Define function to simulate and get L2 data product from each exposure

In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
import gc

def run_one_exposure(label: str, mjd_start: float, seed: int) -> dict:
    """Build one UTR, reduce Stage0–2, return L2 data product."""
    run_dir = OUT
    os.makedirs(run_dir, exist_ok=True)

    ramp, cals, rate_map = create_ramp_from_spectrum(
        OUT,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        fiber_spectra={
            1:(hr8799A_wvs, fiber1_flam),
            2:(hr8799A_wvs, fiber2_flam),
        },
        filter_band="J",  # BSPEC (yJ)
        n_reads=n_reads,
        readtime=READ_TIME,
        seed=seed,
        radial_velocity_km_s=STELLAR_RV, # no dictionary needed since it is the same for both fibers
        barycentric_velocity_km_s=0.0,
        inject_rv_via="spectrum",
        sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
        apply_sky=[1,2,3],
        apply_e2e_throughput=[1,2,3],
        mjd_start=mjd_start-2400000.5,
        save_ramp=False,
        overwrite_ramp=False,
        poisson_noise=True,
        reuse_stage1_sims=True,
        register_stage1_sims=False
    )

    # add fiber keywords
    setattr(ramp.meta, 'object_fiber1', 'SCIENCE')
    setattr(ramp.meta, 'object_fiber2', 'SPECKLE')
    setattr(ramp.meta, 'object_fiber3', 'SKY')

    # add exposure time keyword
    setattr(ramp.meta, 'exposure_time', exposure_time)
    setattr(ramp.meta, 'exposure_type', 'SPEC')

    # set target keywords
    setattr(ramp.meta, 'target_gaia_id', '2832463659640297472')
    setattr(ramp.meta, 'target_name', 'HR 8799')
    setattr(ramp.meta, 'target_ra', 346.8696250)
    setattr(ramp.meta, 'target_dec', 21.1342528)

    wmeta = getattr(cals.get("wavesol"), "meta", None)
    print(
        f"ramp={ramp.meta.filename}  shape={ramp.data.shape}  "
        f"wavesol note={getattr(wmeta, 'note', None)}  "
        f"injected_rv_via={getattr(wmeta, 'injected_rv_via', None)}"
    )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=OUT, dataset=ramp).apply()

    del ramp
    del rate_map

    do_fibers = [str(f) for f in FIBERS]
    # Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
    stage1_args = {
        "spec_extract": {"do_fibers": do_fibers},
        "set_wavesol": {"wavesol": cals["wavesol"]},
    }
    l1 = Stage1Pipeline(output_dir=OUT, primitives=stage1_args, dataset=l0).apply()

    del l0

    injected_m_s = float(STELLAR_RV) * 1000.0
    l2 = Stage2Pipeline(
        output_dir=OUT,
        dataset=l1,
        primitives={
            "blaze_corr": {"do_fibers": do_fibers},
            "stellar_rv": {
                "run": False,
            },
            "sky_sub": {"run": True},
        },
    ).apply()

    del l1
    gc.collect()



Now run

In [ ]:
l2_results = {}

for i in range(total_exposures):
    l2_results[str(obs_jd_arr[i])] = run_one_exposure(label=str(obs_jd_arr[i]),
                                                       mjd_start=obs_jd_arr[i],
                                                       seed=None)